In [1]:
# Import das lib
import sqlite3
import pandas as pd

In [2]:
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

In [3]:
cursor.execute("""
CREATE TABLE funcionarios (
id INTEGER PRIMARY KEY,
nome TEXT NOT NULL,
cargo TEXT NOT NULL,
salario REAL,
data_contratacao TEXT NOT NULL
)
""")

In [8]:
cursor.execute("SELECT * FROM funcionarios;")
pd.DataFrame(cursor.fetchall(), columns = ['id','nome','cargo','salario','data_contratacao'])

,id,nome,cargo,salario,data_contratacao


In [ ]:
cursor.executemany("INSERT INTO funcionarios VALUES(?,?,?,?,?)",[
    (1, "Gustavo", "Analista de Dados", 5.000, "10/2/2025"),
    (2, "Felipe", "Cientista de Dados", 7.000, "11/10/2005"),
    (3, "Beatriz", "Cientista de Dados", 6.000, "16/8/2002"),
    (4, "Giovana", "Analista de Dados", 4.500, "1/5/2004"),
    (5, "Eduardo", "Tecnico de TI", 4.000, "20/3/2006")
])

,id,nome,cargo,salario,data_contratacao


In [11]:
cursor.execute("SELECT * FROM funcionarios")
pd.DataFrame(cursor.fetchall(), columns = ['id','nome','cargo','salario','data_contratacao'])

,id,nome,cargo,salario,data_contratacao
0,1,Gustavo,Analista de Dados,5.0,10/2/2025
1,2,Felipe,Cientista de Dados,7.0,11/10/2005
2,3,Beatriz,Cientista de Dados,6.0,16/8/2002
3,4,Giovana,Analista de Dados,4.5,1/5/2004
4,5,Eduardo,Tecnico de TI,4.0,20/3/2006


In [14]:
cursor.execute("SELECT * FROM funcionarios WHERE salario > 5.000")
pd.DataFrame(cursor.fetchall(), columns = ['id','nome','cargo','salario','data_contratacao'])

,id,nome,cargo,salario,data_contratacao
0,2,Felipe,Cientista de Dados,7.0,11/10/2005
1,3,Beatriz,Cientista de Dados,6.0,16/8/2002


In [16]:
cursor.execute("SELECT * FROM funcionarios WHERE cargo = 'Tecnico de TI'")
pd.DataFrame(cursor.fetchall(), columns = ['id','nome','cargo','salario','data_contratacao'])

,id,nome,cargo,salario,data_contratacao
0,5,Eduardo,Tecnico de TI,4.0,20/3/2006


In [ ]:
cursor.execute("UPDATE funcionarios SET salario = 7.000 WHERE nome = 'Gustavo'")

,id,nome,cargo,salario,data_contratacao


In [24]:
cursor.execute("SELECT * FROM funcionarios WHERE nome = 'Gustavo'")
pd.DataFrame(cursor.fetchall(), columns = ['id','nome','cargo','salario','data_contratacao'])

,id,nome,cargo,salario,data_contratacao
0,1,Gustavo,Analista de Dados,7.0,10/2/2025


In [ ]:
cursor.execute("DELETE FROM funcionarios WHERE nome = 'Giovana'")
conn.commit()

In [35]:
cursor.execute("SELECT COUNT(*) FROM funcionarios GROUP BY nome;")
cursor.fetchall()

[(1,), (1,), (1,), (1,)]

In [47]:
cursor.execute("SELECT SUM(salario), AVG(salario), MAX(salario), MIN(salario) FROM funcionarios")
cursor.fetchall()

[(24.0, 6.0, 7.0, 4.0)]

In [48]:
cursor.execute("SELECT cargo,AVG(salario), COUNT(*) FROM funcionarios GROUP BY cargo")
cursor.fetchall()

[('Analista de Dados', 7.0, 1),
 ('Cientista de Dados', 6.5, 2),
 ('Tecnico de TI', 4.0, 1)]

In [49]:
cursor.execute("SELECT nome, salario FROM funcionarios ORDER BY salario DESC")
cursor.fetchall()

[('Gustavo', 7.0), ('Felipe', 7.0), ('Beatriz', 6.0), ('Eduardo', 4.0)]

In [50]:
cursor.execute("SELECT nome, salario FROM funcionarios ORDER BY nome ASC")
cursor.fetchall()

[('Beatriz', 6.0), ('Eduardo', 4.0), ('Felipe', 7.0), ('Gustavo', 7.0)]

In [51]:
# Exemplo de transação com ROLLBACK
try:
    conn.execute("BEGIN;") # Inicia a trasação
    cursor.execute("INSERT INTO funcionarios VALUES(5, 'Justino', 'Analista de Dados', 5.000, '10/2/2025')") 
    raise Exception("Erro simulado!")  # Força erro
    conn.commit() # Se chegasse aqui, faria commit
except:
    conn.rollback() # Reverte em caso de erro
    print("Transação revertida por erro")

Transação revertida por erro


In [64]:
#  SEM ERRO (COM COMMIT)

try:
    conn.execute("BEGIN;")
    cursor.executemany("INSERT INTO funcionarios VALUES(?,?,?,?,?)",[(6, 'Justino', 'Analista de Dados', 5.000, '10/2/2025')])
    conn.commit()

except:
    conn.rollback()
    print("ROLLBACK")


In [65]:
cursor.execute("SELECT * FROM funcionarios;")
pd.DataFrame(cursor.fetchall(), columns = ['id','nome','cargo','salario','data_contratacao'])

,id,nome,cargo,salario,data_contratacao
0,1,Gustavo,Analista de Dados,7.0,10/2/2025
1,2,Felipe,Cientista de Dados,7.0,11/10/2005
2,3,Beatriz,Cientista de Dados,6.0,16/8/2002
3,5,Eduardo,Tecnico de TI,4.0,20/3/2006
4,6,Justino,Analista de Dados,5.0,10/2/2025


In [66]:
cursor.execute("""
CREATE TABLE departamentos(
id INTEGER PRIMARY KEY,
nome_departamento TEXT NOT NULL,
localizacao TEXT
)
""")

In [68]:
cursor.executemany("INSERT INTO departamentos VALUES (?,?,?)",[
    (1,"Comercial", "Osasco"),
    (2, "Administrativo", "Jandira"),
    (3, "RH", "Carapicúiba")
])

In [69]:
cursor.execute("ALTER TABLE funcionarios ADD COLUMN departamento_id INTEGER")

In [70]:
cursor.execute("UPDATE funcionarios SET departamento_id = 1 WHERE id = '2'")
cursor.execute("UPDATE funcionarios SET departamento_id = 2 WHERE id = '3'")
cursor.execute("UPDATE funcionarios SET departamento_id = 3 WHERE id = '1'")
cursor.execute("UPDATE funcionarios SET departamento_id = 1 WHERE id = '5'")
cursor.execute("UPDATE funcionarios SET departamento_id = 3 WHERE id = '6'")

In [74]:
cursor.execute("SELECT f.nome, d.nome_departamento FROM funcionarios f INNER JOIN departamentos d ON f.departamento_id = d.id")
pd.DataFrame(cursor.fetchall(), columns = ['nome', 'nome_departamento'])

,nome,nome_departamento
0,Gustavo,RH
1,Felipe,Comercial
2,Beatriz,Administrativo
3,Eduardo,Comercial
4,Justino,RH


In [75]:
cursor.execute("""
CREATE TABLE projetos(
id INTEGER PRIMARY KEY,
nome_projeto TEXT NOT NULL
)
""")
cursor.execute("""
CREATE TABLE funcionarios_projetos(
funcionario_id INTEGER,
projetos_id INTEGER
)
""")

cursor.executemany(
    "INSERT INTO projetos VALUES (?, ?)",[
(1, 'Sistema Financeiro'),
(2, 'Aplicativo da Empresa'),
(3, 'Dashboard de Indicadores'),
]
)

cursor.executemany(
"INSERT INTO funcionarios_projetos VALUES (?, ?)",
[(5, 1),
(6, 2),
(1, 2),
(2, 3),
(3, 3),
]
)

conn.commit()

